# ST-GCN Joint — NTU60 xsub — resume to 80 effective epochs

This notebook resumes the supplied epoch-8 checkpoint and trains outer epochs 9–16 with `RepeatDataset(times=5)`. The resulting run has 16 outer epochs and 80 effective dataset passes. It does not repeat smoke tests or batch-size benchmarking.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mzuyyy/Human-action-recognition.git'
PROJECT_DIR = Path('/kaggle/working/ntu-action-recognition')
MMACTION2_DIR = Path('/kaggle/working/mmaction2')
CONFIG_PATH = PROJECT_DIR / 'configs/stgcn_ntu60_xsub_80e_resume.py'
WORK_DIR = PROJECT_DIR / 'work_dirs/stgcn_ntu60_xsub_80e_resume'
MODEL_CHECKPOINT_PATH = Path('/kaggle/input/models/duymaingoc/resume/pytorch/default/1/best_acc_top1_epoch_8.pth')
NTU60_URL = 'https://download.openmmlab.com/mmaction/v1.0/skeleton/data/ntu60_2d.pkl'
RESUME = True  # Required: restore model, optimizer, scheduler and epoch state.

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('experiment work directory:', WORK_DIR)


In [ ]:
%%bash
set -euo pipefail
python -m pip uninstall -q -y mmcv mmcv-lite >/dev/null 2>&1 || true
python -m pip install -q --only-binary=mmcv-lite \
  "importlib-metadata" \
  "mmengine>=0.7.1,<1.0.0" \
  "mmcv-lite==2.1.0"

MMACTION2_SRC=/kaggle/working/mmaction2
if [ ! -d "${MMACTION2_SRC}/.git" ]; then
  git -c advice.detachedHead=false clone --branch v1.2.0 --depth 1 \
    https://github.com/open-mmlab/mmaction2.git "${MMACTION2_SRC}"
else
  if ! git -C "${MMACTION2_SRC}" rev-parse -q --verify "refs/tags/v1.2.0^{commit}" >/dev/null; then
    git -C "${MMACTION2_SRC}" fetch -q --depth 1 origin tag v1.2.0
  fi
  git -c advice.detachedHead=false -C "${MMACTION2_SRC}" checkout -q --detach v1.2.0
fi
python -m pip uninstall -q -y mmaction2 >/dev/null 2>&1 || true
python -m pip install -q -e "${MMACTION2_SRC}"

# The joint-only experiment does not use ViNLU. Prevent MMAction2 1.2.0 from
# importing its old optional multimodal code against Kaggle's new Transformers.
python - <<'PY'
from pathlib import Path
path = Path('/kaggle/working/mmaction2/mmaction/utils/dependency.py')
text = path.read_text()
old = "WITH_MULTIMODAL = all(\n    satisfy_requirement(item) for item in ['transformers>=4.28.0'])"
new = "# Disabled for this skeleton-only environment.\nWITH_MULTIMODAL = False"
if old in text:
    path.write_text(text.replace(old, new))
elif new not in text:
    raise RuntimeError(f'Could not disable MMAction2 multimodal imports in {path}')
PY


In [ ]:
# Fetch the already-approved NTU60 2D annotations; no repeated inspection.
import glob, os, urllib.request

dst = PROJECT_DIR / 'data/skeleton/ntu60_2d.pkl'
if dst.exists():
    print('dataset ready:', dst)
else:
    hits = glob.glob('/kaggle/input/**/ntu60_2d.pkl', recursive=True)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if hits:
        os.symlink(os.path.abspath(hits[0]), dst)
        print('linked dataset:', hits[0])
    else:
        temporary = dst.with_suffix(dst.suffix + '.part')
        temporary.unlink(missing_ok=True)
        try:
            urllib.request.urlretrieve(NTU60_URL, temporary)
            temporary.replace(dst)
        except Exception:
            temporary.unlink(missing_ok=True)
            raise
        print('downloaded dataset:', dst)


In [ ]:
# Record the complete resolved configuration before launching training.
# Editable installs are activated only in new Python processes; expose both
# source trees explicitly to this already-running Papermill kernel.
import importlib, shutil, sys
for source_dir in (PROJECT_DIR, MMACTION2_DIR):
    source = str(source_dir.resolve())
    if source not in sys.path:
        sys.path.insert(0, source)
importlib.invalidate_caches()
from mmengine.config import Config

if not RESUME:
    raise RuntimeError('This notebook is intentionally a resume run; keep RESUME=True.')
last_checkpoint_marker = WORK_DIR / 'last_checkpoint'
if last_checkpoint_marker.is_file():
    RESUME_CHECKPOINT = Path(last_checkpoint_marker.read_text().strip())
    if not RESUME_CHECKPOINT.is_absolute():
        RESUME_CHECKPOINT = WORK_DIR / RESUME_CHECKPOINT
else:
    RESUME_CHECKPOINT = MODEL_CHECKPOINT_PATH
if not RESUME_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f'Attach the Kaggle Model containing: {RESUME_CHECKPOINT}')
source_best_checkpoint = WORK_DIR / 'best_acc_top1_epoch_8.pth'
if MODEL_CHECKPOINT_PATH.is_file() and not source_best_checkpoint.exists():
    shutil.copy2(MODEL_CHECKPOINT_PATH, source_best_checkpoint)
    print('preserved source best checkpoint:', source_best_checkpoint)

resolved_cfg = Config.fromfile(str(CONFIG_PATH))
resolved_cfg.work_dir = str(WORK_DIR)
resolved_path = WORK_DIR / 'resolved_config.py'
resolved_cfg.dump(str(resolved_path))
print('resolved config:', resolved_path)
print('resume checkpoint:', RESUME_CHECKPOINT)
print('outer epochs:', resolved_cfg.train_cfg.max_epochs)
print('dataset repeats:', resolved_cfg.train_dataloader.dataset.times)
print('effective epochs:', resolved_cfg.train_cfg.max_epochs * resolved_cfg.train_dataloader.dataset.times)
print('batch size / learning rate:', resolved_cfg.train_dataloader.batch_size, '/', resolved_cfg.optim_wrapper.optimizer.lr)


In [ ]:
# Resume the complete MMEngine training state and run outer epochs 9–16.
import json, os, subprocess, sys, time

def run_training():
    command = [
        sys.executable, str(MMACTION2_DIR / 'tools/train.py'), str(CONFIG_PATH),
        '--work-dir', str(WORK_DIR), '--seed', '42',
        '--resume', str(RESUME_CHECKPOINT)]

    train_env = os.environ.copy()
    train_env['PYTHONPATH'] = (
        str(MMACTION2_DIR) + os.pathsep + str(PROJECT_DIR) + os.pathsep
        + train_env.get('PYTHONPATH', ''))
    train_env['PYTHONUNBUFFERED'] = '1'
    console_path = WORK_DIR / 'training_console.log'
    started = time.time()
    with console_path.open('a', buffering=1) as console:
        process = subprocess.Popen(
            command, cwd=str(PROJECT_DIR), env=train_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            console.write(line)
        return_code = process.wait()
    elapsed = time.time() - started
    (WORK_DIR / 'training_run.json').write_text(json.dumps({
        'command': command, 'resume': True,
        'resume_checkpoint': str(RESUME_CHECKPOINT),
        'wall_time_sec': elapsed,
        'return_code': return_code
    }, indent=2))
    if return_code != 0:
        raise RuntimeError(f'training failed with return code {return_code}')
    print(f'training completed in {elapsed / 3600:.2f} hours')

run_training()


In [ ]:
# Report the resumed phase (outer epochs 9–16) and final 80-pass state.
import json, re

metrics_path = WORK_DIR / 'epoch_metrics.jsonl'
if not metrics_path.is_file():
    raise FileNotFoundError(f'training metrics not found: {metrics_path}')
raw_records = [
    json.loads(line) for line in metrics_path.read_text().splitlines()
    if line.strip()]
records_by_epoch = {
    int(record['outer_epoch']): record for record in raw_records
    if 9 <= int(record['outer_epoch']) <= 16}
expected_epochs = set(range(9, 17))
missing_epochs = sorted(expected_epochs - records_by_epoch.keys())
if missing_epochs:
    raise RuntimeError(
        f'resumed metrics are incomplete; missing outer epochs: {missing_epochs}')
records = [records_by_epoch[epoch] for epoch in range(9, 17)]

table_lines = [
    '| Outer Epoch | Effective Epoch | Train Loss | Top-1 | Top-5 |',
    '|-------------|-----------------|------------|-------|-------|']
def metric_text(value, spec):
    return 'unavailable' if value is None else format(value, spec)

for record in records:
    table_lines.append(
        f"| {record['outer_epoch']} | {record['effective_epoch']} | "
        f"{metric_text(record['train_loss'], '.6f')} | "
        f"{metric_text(record['val_acc_top1'], '.4f')} | "
        f"{metric_text(record['val_acc_top5'], '.4f')} |")
table = '\n'.join(table_lines)

final = records[-1]
total_wall = sum(item['epoch_wall_time_sec'] for item in records)
peak_gpu = max(item['gpu_memory_mb'] for item in records)
final_checkpoint = WORK_DIR / 'epoch_16.pth'
if not final_checkpoint.is_file():
    raise FileNotFoundError(f'completed checkpoint not found: {final_checkpoint}')

def checkpoint_epoch(path):
    match = re.search(r'epoch_(\d+)\.pth$', path.name)
    return int(match.group(1)) if match else -1

new_best_checkpoints = [
    path for path in WORK_DIR.glob('best_acc_top1_epoch_*.pth')
    if checkpoint_epoch(path) in records_by_epoch]
if new_best_checkpoints:
    best_checkpoint = max(
        new_best_checkpoints,
        key=lambda path: records_by_epoch[checkpoint_epoch(path)]['val_acc_top1'])
    best = records_by_epoch[checkpoint_epoch(best_checkpoint)]
    best_issue = 'None.'
else:
    best_checkpoint = source_best_checkpoint
    best = dict(
        outer_epoch=8, effective_epoch=40,
        val_acc_top1=None, val_acc_top5=None)
    best_issue = (
        'No epoch 9–16 checkpoint beat the resumed epoch-8 best. The source '
        'checkpoint remains best; its validation values were not bundled '
        'with the checkpoint-only Kaggle Model.')
if not best_checkpoint.is_file():
    raise FileNotFoundError(f'best checkpoint not found: {best_checkpoint}')
latest_checkpoint = WORK_DIR / 'latest.pth'
temporary_link = WORK_DIR / '.latest.pth.tmp'
temporary_link.unlink(missing_ok=True)
temporary_link.symlink_to(final_checkpoint.name)
temporary_link.replace(latest_checkpoint)

report = f"""EXPERIMENT
----------
Model: ST-GCN
Dataset: NTU60 2D skeleton
Split: Cross-Subject (xsub_train / xsub_val)
Input: Joint, COCO-17
Outer epochs: 16 (resumed from epoch 8)
RepeatDataset: 5
Effective epochs: 80
Batch size: 64

TRAINING
--------
Total training wall time: {metric_text(total_wall, '.1f')} seconds (epochs 9–16)
Final training loss: {metric_text(final['train_loss'], '.6f')}
Peak GPU memory: {metric_text(peak_gpu, '.0f')} MiB
Final learning rate: {metric_text(final['learning_rate'], '.10g')}

BEST VALIDATION
---------------
Best outer epoch: {best['outer_epoch']}
Effective epoch: {best['effective_epoch']}
Top-1: {metric_text(best['val_acc_top1'], '.4f')}
Top-5: {metric_text(best['val_acc_top5'], '.4f')}

FINAL VALIDATION
----------------
Top-1: {metric_text(final['val_acc_top1'], '.4f')}
Top-5: {metric_text(final['val_acc_top5'], '.4f')}

CHECKPOINTS
-----------
Best checkpoint: {best_checkpoint}
Latest checkpoint: {latest_checkpoint}
Final checkpoint: {final_checkpoint}

OUTPUTS
-------
Config: {WORK_DIR / 'resolved_config.py'}
Logs: {WORK_DIR}/<timestamp>/vis_data/scalars.json
Work directory: {WORK_DIR}

ISSUES
------
{best_issue}
"""
full_report = table + '\n\n' + report
report_path = PROJECT_DIR / 'artifacts/stgcn_ntu60_xsub_80e_resume_report.txt'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(full_report)
print(full_report)
print('report:', report_path)
